In [20]:
##modules
#%matplotlib widget
#%matplotlib inline

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
# #

import numpy as np
# np.show_config()
%matplotlib qt

import mne
mne.set_log_level("ERROR")
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('QtAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs
from statsmodels.tsa.stattools import acf

import numpy as np

import pandas as pd

from scipy.stats import permutation_test

from statsmodels.tsa.stattools import acf

sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")



# Ahora importa la función
from print5 import print5
from more_itertools import collapse
from joblib import Parallel, delayed




from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")



import pickle

from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt

# Import required code for visualizing example models
from fooof import FOOOF
from fooof.sim.gen import gen_power_spectrum
from fooof.sim.utils import set_random_seed
from fooof.plts.spectra import plot_spectra
from fooof.plts.annotate import plot_annotated_model
from fooof import FOOOFGroup,Bands
from fooof.analysis.periodic import get_band_peak_group, get_band_peak

import gc



In [21]:
sys.path.append("..")  # esto sube un nivel desde Scripts_visual_block


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "event"
if modality=="auditory":
    subj="sub-A2002"
else:
    subj = "sub-V1001"

# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")

mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)

with open(datadir / f"subjects_remove_{modality}.pkl", "rb") as f:
    subjects_remove = pickle.load(f)
    



📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_event
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\ICA_event
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_event
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_event\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_event\fwd
inverse_pat

In [22]:

#epochs


subjects=[]

# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subjects.append(subdirectorio.name)

print(subjects)



##tablas de canales

# channels_path = channels_structure_path   / f"channels_mag_{modality}.csv"

# channels = pd.read_csv(channels_path)
# channels_mag=channels[channels[f"canal_efectivo"].notna()][f"canal_efectivo"]
# channels_mag=channels_mag.tolist()
# del channels
# print(len(channels_mag))

# epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{subjects[0]}_epochs_zinnen_{layer_script}-epo.fif")
window_size=3
sliding_window=0.351
lfreq=1
hfreq=40


filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")
  

#suffix for saving the tables
suffix = [layer_script]
if filtering:
    suffix.insert(0, filter_name)
suffix_str = "_".join(suffix)


log_dir = ACW_path / "logs_dynamic_fooof"
log_dir.mkdir(parents=True, exist_ok=True)

    
freq_bands = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
    'gamma': (30, 40)
}  

bands = Bands(freq_bands)

aperiodic_mode="fixed" # "fixed" or "knee"

# fmin=1
# fmax=40

# select_channels="only_zinnen"

# condition="zinnen"
# condition_epoch=f"fix_{condition.upper()}"
return_fg_subject=True

select_channels=None
select_highest=True


if layer_script=="block":
    change_name="fix"
elif layer_script=="event":
    change_name="begin"
    
    
    
    

['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

In [23]:
from pathlib import Path

log_dir = Path(log_dir)

# crear carpeta backup
backup_dir = log_dir / "back_up_fooof"
backup_dir.mkdir(exist_ok=True)

# listar archivos .txt y .pickle
files = [
    p.name
    for p in log_dir.iterdir()
    if p.is_file() and p.suffix in (".txt", ".pickle")
]


# guardar lista en un txt
backup_file = backup_dir / "files_in_log_dir.txt"

#si ya existe, borralo, asi tenemos la lista actualizada
if backup_file.exists():
    backup_file.unlink()

with open(backup_file, "w", encoding="utf-8") as f:
    for fname in sorted(files):
        f.write(f"{fname}\n")

In [5]:
def check_nans(data, nan_policy='zero'):
    """Check an array for nan values, and replace, based on policy."""

    # Find where there are nan values in the data
    nan_inds = np.where(np.isnan(data))

    # Apply desired nan policy to data
    if nan_policy == 'zero':
        data[nan_inds] = 0
    elif nan_policy == 'mean':
        data[nan_inds] = np.nanmean(data)
    else:
        raise ValueError('Nan policy not understood.')

    return data

In [6]:
dict_isc= pd.read_pickle(ISC_block_path /f"ISC_results_block.pkl")
dict_woorden_block=dict_isc['dict_isc_WOORDEN']
dict_zinnen_block = dict_isc['dict_isc_ZINNEN']
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted"]
print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

names_channels_woorden=dict_woorden_block["significant_channels_adjusted_names"]
names_channels_zinnen=dict_zinnen_block["significant_channels_adjusted_names"]

print("names_channels_woorden", names_channels_woorden)
print("names_channels_zinnen", names_channels_zinnen)

h_subj =0
path_epochs = epochs_clean_path / f"{subjects[h_subj]}_epochs_{layer_script}-epo.fif"
epochs = mne.read_epochs(path_epochs, preload=False)
#establecimiento de canales palabras, canales frases y canales mixtos
# Tomamos el orden original de los canales del objeto epochs
all_channels = epochs.ch_names
all_channels_numbers= [epochs.ch_names.index(ch) for ch in epochs.ch_names]


del epochs

# Palabras
numbers_channels_only_woorden = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch not in numbers_channels_zinnen
]

numbers_channels_only_zinnen = [
    ch for ch in all_channels_numbers if ch in numbers_channels_zinnen and ch not in numbers_channels_woorden
]

numbers_channels_intersection = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch in numbers_channels_zinnen
]

# Lo mismo pero usando nombres
names_channels_only_woorden = [
    ch for ch in all_channels if ch in names_channels_woorden and ch not in names_channels_zinnen
]

names_channels_only_zinnen = [
    ch for ch in all_channels if ch in names_channels_zinnen and ch not in names_channels_woorden
]

names_channels_intersection = [
    ch for ch in all_channels if ch in names_channels_woorden and ch in names_channels_zinnen
]
# ---------------------------
# Print resumen
# ---------------------------

print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)}, "
      f"len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)}, "
      f"len(significant_channels_intersection): {len(numbers_channels_intersection)}")



print("\n✅ Only woorden (names):", names_channels_only_woorden)
print("✅ Only zinnen (names):", names_channels_only_zinnen)
print("✅ Intersection (names):", names_channels_intersection)


dict_select_channels={
    "names_channels_only_woorden": names_channels_only_woorden,
    "names_channels_only_zinnen": names_channels_only_zinnen,
    "names_channels_intersection": names_channels_intersection
}

numbers_channels_woorden [  2   3   4   5   7   8   9  10  11  12  13  14  17  18  35  36  40  41
  42  43  45  46  47  48  49  51  52  53  54  56  57  58  60  61  62  64
  65  66  68  69  70  73  75  77  78  80  81  82  83  84  85  86  87  88
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 131 132 133 134 135 136 137 138 139 140 141 142 143 144 146 147
 148 149 150 151 182 183 184 188 192 196 199 200 201 202 204 206 207 209
 210 211 212 214 215 216 217 221 222 223 224 225 226 228 229 230 231 232
 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250
 251 252 253 254 255 256 257 258] numbers_channels_zinnen [  2   3   4   5   7   8   9  10  11  12  13  14  17  18  19  24  25  26
  29  30  31  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47
  48  49  51  52  53  54  55  56  57  58  59  62  66  75  77  78  79  80
  81  82  83  84  85  86  87  88  89  90 

In [7]:
def dynamic_FOOOF_epochs(subj,window_size,sliding_window, bands,dict_select_channels,lfreq, hfreq,change_name=None ,aperiodic_mode="fixed", select_highest=False, select_channels=None,  filtering=False, filter_name=None):

    """
    this function calculates the FOOOF dynamically for the whole epoch object for each condition, epoch and electrode

    then it dvides each resulting segment, in different sliding windows, obtainin a result for each window

    It also computes an slope and std metric to the the differences through each segment, giving a dynamic metric


    it works basically decomposing for condition, epoch and electrode, using nestted lists, 

    e.g: delta_window_all_elect_all_epoch_all  : you can see that the level of the list goes backwards, 

    - window_all means it contains the values for all sliding windows for each segment

    -- instead of window_all  there can be slope or std, wich means that the slope or std is calculated thoughout the whole sliding windows


    - elect_all: contains all values for each elect
    ...

    if only elect appears instead of elect_all , it means it only takes the value for ONE electrode

    delta_window_all_elect_epoch   : this would only take all values for sliding windows in one epoch and electrode


    """
    
    #-------------------------------------------------------------------------
    # function for computing psd and FOOOF
    def compute_psd_FOOOF(segment, sfreq, fmin, fmax, n_per_seg,aperiodic_mode, freq_resolution ):
        
        #calculates psd 
                        
        psd, freqs = mne.time_frequency.psd_array_welch(
            segment[np.newaxis, :],
            sfreq=sfreq,
            fmin=fmin,
            fmax=fmax,
            n_per_seg=n_per_seg,
            # n_fft=n_per_seg,
            # n_jobs=10
        )
        
        
        # Creamos FOOOFGroup para este segmento
        fg_segment = FOOOF(
            peak_width_limits=[2*freq_resolution, 12],
            max_n_peaks=4,
            min_peak_height=0.2,
            peak_threshold=2.0,
            aperiodic_mode=aperiodic_mode,   # 'fixed' o 'knee'
        )
        
        fg_segment.fit(freqs, psd[0],)
        

        return fg_segment
    #-------------------------------------------------------------------------
    
    
    
    #suffix for saving the tables
    suffix = [layer_script]
    if filtering:
        suffix.insert(0, filter_name)
    suffix_str = "_".join(suffix)


    log_file = log_dir/f"{subj}_log_dynamic_fooof_{suffix_str}.txt"

    with open(log_file, "a") as f:
        f.write(f"{subj}: starting processing\n")
        
    try:        
        path_epochs = epochs_clean_path / f"{subj}_epochs_{layer_script}-epo.fif"
        
        # read epochs
        epochs = mne.read_epochs(path_epochs)
        if len(epochs) == 0:
            with open(log_file, "a") as f:
                f.write(f"{subj}: NO epochs available, skipping subject\n")
            return None, None
        
        
        # ---- LOG DE CONDICIONES PRESENTES ----
        condition_names = list(epochs.event_id.keys())

        with open(log_file, "a") as f:
            f.write(f"{subj}: CONDITIONS_PRESENT ({len(condition_names)}):\n")
            for cond in condition_names:
                try:
                    n_ep = len(epochs[cond])
                except Exception:
                    n_ep = "NA"
                f.write(f"    - {cond}: {n_ep} epochs\n")
        
        
        bands = Bands(freq_bands)
        
        with open(log_file, "a") as f:
            f.write(f"{subj}: epochs loaded (n_epochs={len(epochs)})\n")
        # ✅ CREAR bands DENTRO del proceso
        with open(log_file, "a") as f:
            f.write(f"{subj}: frequency bands created\n")

        #filter epochs
        if filtering:
            epochs.filter(l_freq=lfreq, h_freq=hfreq, n_jobs=1)
            with open(log_file, "a") as f:
                f.write(f"{subj}: filtering applied ({lfreq}-{hfreq} Hz)\n")


        

        #get duration
        duration= epochs.tmax - epochs.tmin
        #get sample frequency
        sfreq= epochs.info['sfreq']


        condition_names = list(epochs.event_id.keys())

        # Convertir a muestras
        sliding_window_samples = int(sliding_window * sfreq)
        window_size_samples = int(window_size * sfreq)
        


        for i_cond in range(0,len(condition_names)):   
        # for i_cond in range(0,1):   
            
            #now i take the original name to select the epochs
            cond=condition_names[i_cond]
            
            if change_name:
                cond_clean=cond.removeprefix(f"{change_name}_")
            else:
                cond_clean=cond
            
                
             # ---------------------------------------------------------
            # CHECK EXISTING TABLES FOR THIS CONDITION
            # ---------------------------------------------------------
            table_dynamic_path = log_dir / f"{subj}_{cond}_table_dynamic_fooof_all_{suffix_str}.pickle"
            table_results_path = log_dir / f"{subj}_{cond}_table_dynamic_fooof_results_all_{suffix_str}.pickle"

            dynamic_exists = table_dynamic_path.exists()
            results_exists = table_results_path.exists()

            # ✅ If BOTH tables exist → skip condition
            if dynamic_exists and results_exists:
                continue

            # ⚠️ If ONLY ONE exists → delete it and recompute condition
            if dynamic_exists ^ results_exists:  # XOR: solo una existe


                if dynamic_exists:
                    try:
                        table_dynamic_path.unlink()
                    except FileNotFoundError:
                        pass

                if results_exists:
                    try:
                        table_results_path.unlink()
                    except FileNotFoundError:
                        pass  
 
            #now i select the epochs for that condition
            epochs_cond = epochs[cond]
            
            if len(epochs_cond) == 0:
                with open(log_file, "a") as f:
                    f.write(
                        f"{subj}: NO epochs available for condition {cond}, skipping condition\n"
                    )
                continue
                                

            # select channels where you are applying the filter
            if select_channels:
                
                picks= dict_select_channels[f"names_channels_{select_channels}"]
                
                epochs_filt = epochs_cond.pick(picks)
            elif select_channels==None:
                epochs_filt=epochs_cond        

            
            del epochs_cond
                      
            
            gc.collect()
                        # channels = epochs_filt.ch_names
            channels_mag = epochs_filt.pick(picks="meg", exclude="bads").ch_names       

            data_epochs = epochs_filt.get_data()   # (n_epochs, n_channels, n_times)
            
            fmin=epochs_filt.info['highpass']
            fmax=epochs_filt.info['lowpass']
            
            del epochs_filt
            gc.collect()
            
            #lower oscillation is the inverse of the lowest frequency, multiplied by 3 to ensure at least 3 cycles are captured
            lower_oscillation=(1/fmin)*3
            
            
            n_per_seg = int(lower_oscillation * sfreq)

            freq_resolution=sfreq/n_per_seg
            
            assert window_size_samples >= n_per_seg, (
                    f"window_size ({window_size}s) too short for fmin={fmin}Hz"
            )

            n_epochs, n_channels, n_times = data_epochs.shape
            
            

            ### VALORES POR OBJETO EPOCH * ELECT * WINDOW (IF THERE IS WINDOW)
            
            ## VALORES POR VENTANA WINDOW ALL

            # ===== BANDAS: valores por ventana =====
            delta_window_all_elect_all_epoch_all = []
            theta_window_all_elect_all_epoch_all = []
            alpha_window_all_elect_all_epoch_all = []
            beta_window_all_elect_all_epoch_all  = []
            gamma_window_all_elect_all_epoch_all = []

            # ===== APERIODIC: valores por ventana =====
            offsets_window_all_elect_all_epoch_all = []
            exponents_window_all_elect_all_epoch_all = []

            # ===== MÉTRICAS DEL AJUSTE: valores por ventana =====
            r2_window_all_elect_all_epoch_all = []
            error_window_all_elect_all_epoch_all = []

            # ===== KNEE: valor por ventana (si aplica) =====
            if aperiodic_mode == "knee":
                knee_window_all_elect_all_epoch_all = []
                
            ### VALORES SLOPE AND STD
                
            # ===== BANDAS: slope y std =====
            delta_slope_elect_all_epoch_all = []
            delta_std_elect_all_epoch_all   = []

            theta_slope_elect_all_epoch_all = []
            theta_std_elect_all_epoch_all   = []

            alpha_slope_elect_all_epoch_all = []
            alpha_std_elect_all_epoch_all   = []

            beta_slope_elect_all_epoch_all  = []
            beta_std_elect_all_epoch_all    = []

            gamma_slope_elect_all_epoch_all = []
            gamma_std_elect_all_epoch_all   = []

            # ===== APERIODIC: slope y std =====
            offsets_slope_elect_all_epoch_all = []
            offsets_std_elect_all_epoch_all   = []

            exponents_slope_elect_all_epoch_all = []
            exponents_std_elect_all_epoch_all   = []

            # ===== KNEE: slope y std (si aplica) =====
            if aperiodic_mode == "knee":
                knee_slope_elect_all_epoch_all = []
                knee_std_elect_all_epoch_all   = []

        

            ##for each epoch  
            for j_epoch in range(0,len(data_epochs)):
            # for j_epoch in range(0,1):   
                #take data for each epoch
                data_epoch=data_epochs[j_epoch]


                ## VALORES POR VENTANA WINDOW ALL
                
                # ===== BANDAS: valores por ventana =====
                delta_window_all_elect_all_epoch = []
                theta_window_all_elect_all_epoch = []
                alpha_window_all_elect_all_epoch = []
                beta_window_all_elect_all_epoch  = []
                gamma_window_all_elect_all_epoch = []

                # ===== APERIODIC: valores por ventana =====
                offsets_window_all_elect_all_epoch = []
                exponents_window_all_elect_all_epoch = []

                # ===== MÉTRICAS DEL AJUSTE: valores por ventana =====
                r2_window_all_elect_all_epoch = []
                error_window_all_elect_all_epoch = []

                # ===== KNEE: valor por ventana =====
                if aperiodic_mode == "knee":
                    knee_window_all_elect_all_epoch = []
                    
                    
                ## VALORES POR SLOPE Y STD

                # ===== BANDAS: slope y std =====
                delta_slope_elect_all_epoch = []
                delta_std_elect_all_epoch   = []

                theta_slope_elect_all_epoch = []
                theta_std_elect_all_epoch   = []

                alpha_slope_elect_all_epoch = []
                alpha_std_elect_all_epoch   = []

                beta_slope_elect_all_epoch  = []
                beta_std_elect_all_epoch    = []

                gamma_slope_elect_all_epoch = []
                gamma_std_elect_all_epoch   = []

                # ===== APERIODIC: slope y std =====
                offsets_slope_elect_all_epoch = []
                offsets_std_elect_all_epoch   = []

                exponents_slope_elect_all_epoch = []
                exponents_std_elect_all_epoch   = []

                # ===== KNEE: slope y std =====
                if aperiodic_mode == "knee":
                    knee_slope_elect_all_epoch = []
                    knee_std_elect_all_epoch   = []
                    
                #calculus of value for EACH SENSOR
                for h_elect in range(0,len(data_epoch)):
                # for h_elect in range(0,1):
        
                    ###break epochs in segments of size window_size_samples with a step of sliding_window_samples
                    epoch = data_epoch[h_elect]
                    n_samples = epoch.shape[0]
                    

                    # listas FOOOF por CANAL (dentro de esta época) — sin slopes/std aquí
                    delta_window_all_elect_epoch = []
                    theta_window_all_elect_epoch = []
                    alpha_window_all_elect_epoch = []
                    beta_window_all_elect_epoch  = []
                    gamma_window_all_elect_epoch = []

                    offsets_window_all_elect_epoch   = []
                    exponents_window_all_elect_epoch = []

                    r2_window_all_elect_epoch    = []
                    error_window_all_elect_epoch = []

                    # solo si aperiodic_mode == "knee"
                    if aperiodic_mode == "knee":
                        knee_window_all_elect_epoch = []
                    
                    
                    # 2️⃣ Crear lista de segmentos válidos
                    segments = [
                        epoch[start:start + window_size_samples]
                        for start in range(0, n_samples - window_size_samples + 1, sliding_window_samples)
                        if epoch[start:start + window_size_samples].shape[0] == window_size_samples
                    ]

                    window_number = len(segments)
                    
                    # 3️⃣ Procesar en paralelo con joblib
                    ##compute_psd_FOOOF(segment, sfreq, fmin, fmax, n_per_seg,aperiodic_mode 
                    
                    window_results = Parallel(n_jobs=1)(
                        delayed(compute_psd_FOOOF)(
                            segment, sfreq, fmin, fmax, n_per_seg,aperiodic_mode, freq_resolution=freq_resolution
                        )
                        for segment in segments
                    )
                    
                                
                    del segments
                    gc.collect()

                    # 4️⃣ Desempaquetar resultados, valores por WINDOW_ALL, (NO hay slopes ni std aqui)
                    for fg_segment in window_results:

                        ##parámetros periodicos
                        for label, definition in bands:

                            band_ps = get_band_peak(
                                fg_segment.peak_params_,
                                definition,
                                select_highest=select_highest
                            )

                            if select_highest is False:
                                if band_ps.size > 3:
                                    total_power = np.sum(band_ps[:, 1])
                                else:
                                    band_ps = check_nans(band_ps)
                                    total_power = band_ps[1]
                            else:
                                band_ps = check_nans(band_ps)
                                total_power = band_ps[1]
                                
                                

                            if label == "delta":
                                delta_window_all_elect_epoch.append(total_power)
                            elif label == "theta":
                                theta_window_all_elect_epoch.append(total_power)
                            elif label == "alpha":
                                alpha_window_all_elect_epoch.append(total_power)
                            elif label == "beta":
                                beta_window_all_elect_epoch.append(total_power)
                            elif label == "gamma":
                                gamma_window_all_elect_epoch.append(total_power)
                        

                                
                        # ===== aperiodic parameters (por ventana) =====
                        aperiodic_params = fg_segment.get_params('aperiodic_params')

                        if aperiodic_mode == "fixed":
                            offsets_window_all_elect_epoch.append(aperiodic_params[0])
                            exponents_window_all_elect_epoch.append(aperiodic_params[1])

                        elif aperiodic_mode == "knee":
                            offsets_window_all_elect_epoch.append(aperiodic_params[0])
                            knee_window_all_elect_epoch.append(aperiodic_params[1])
                            exponents_window_all_elect_epoch.append(aperiodic_params[2])


                        # ===== métricas del ajuste (por ventana) =====
                        r2_window_all_elect_epoch.append(fg_segment.get_params('r_squared'))
                        error_window_all_elect_epoch.append(fg_segment.get_params('error'))
                        
                    #check for inconsistensies
                    if len(delta_window_all_elect_epoch) != window_number:
                        raise ValueError(
                            f"Electrode {h_elect}, epoch {j_epoch}: "
                            f"{len(delta_window_all_elect_epoch)} != {window_number}"
                        )
                                

                    del window_results
                    gc.collect()

                    
                    ### ADD RESULT TO ALL SENSORS  (FOOOF)
                    
                    ## VALORES POR WINDOW ALL

                    # ===== guardar series por electrodo (ventanas) =====
                    delta_window_all_elect_all_epoch.append(delta_window_all_elect_epoch)
                    theta_window_all_elect_all_epoch.append(theta_window_all_elect_epoch)
                    alpha_window_all_elect_all_epoch.append(alpha_window_all_elect_epoch)
                    beta_window_all_elect_all_epoch.append(beta_window_all_elect_epoch)
                    gamma_window_all_elect_all_epoch.append(gamma_window_all_elect_epoch)

                    offsets_window_all_elect_all_epoch.append(offsets_window_all_elect_epoch)
                    exponents_window_all_elect_all_epoch.append(exponents_window_all_elect_epoch)
                    
                    # ===== métricas del ajuste: SOLO valor =====
                    r2_window_all_elect_all_epoch.append(r2_window_all_elect_epoch)
                    error_window_all_elect_all_epoch.append(error_window_all_elect_epoch)
                    
                    # ===== knee: solo si existe (valor + slope + std) =====
                    if aperiodic_mode == "knee":
                        knee_window_all_elect_all_epoch.append(knee_window_all_elect_epoch)

                    ## ===== slope + std POR ELECTRODO (en esta época) =====

                    # delta
                    x = range(len(delta_window_all_elect_epoch))
                    if len(delta_window_all_elect_epoch) >= 2:
                        delta_slope_elect_all_epoch.append(np.polyfit(x, delta_window_all_elect_epoch, 1)[0])
                    else:
                        delta_slope_elect_all_epoch.append(np.nan)
                    delta_std_elect_all_epoch.append(np.std(delta_window_all_elect_epoch))

                    # theta
                    x = range(len(theta_window_all_elect_epoch))
                    if len(theta_window_all_elect_epoch) >= 2:
                        theta_slope_elect_all_epoch.append(np.polyfit(x, theta_window_all_elect_epoch, 1)[0])
                    else:
                        theta_slope_elect_all_epoch.append(np.nan)
                    theta_std_elect_all_epoch.append(np.std(theta_window_all_elect_epoch))

                    # alpha
                    x = range(len(alpha_window_all_elect_epoch))
                    if len(alpha_window_all_elect_epoch) >= 2:
                        alpha_slope_elect_all_epoch.append(np.polyfit(x, alpha_window_all_elect_epoch, 1)[0])
                    else:
                        alpha_slope_elect_all_epoch.append(np.nan)
                    alpha_std_elect_all_epoch.append(np.std(alpha_window_all_elect_epoch))

                    # beta
                    x = range(len(beta_window_all_elect_epoch))
                    if len(beta_window_all_elect_epoch) >= 2:
                        beta_slope_elect_all_epoch.append(np.polyfit(x, beta_window_all_elect_epoch, 1)[0])
                    else:
                        beta_slope_elect_all_epoch.append(np.nan)
                    beta_std_elect_all_epoch.append(np.std(beta_window_all_elect_epoch))

                    # gamma
                    x = range(len(gamma_window_all_elect_epoch))
                    if len(gamma_window_all_elect_epoch) >= 2:
                        gamma_slope_elect_all_epoch.append(np.polyfit(x, gamma_window_all_elect_epoch, 1)[0])
                    else:
                        gamma_slope_elect_all_epoch.append(np.nan)
                    gamma_std_elect_all_epoch.append(np.std(gamma_window_all_elect_epoch))


                    # ===== aperiodic slopes and std =====

                    # offsets
                    x = range(len(offsets_window_all_elect_epoch))
                    if len(offsets_window_all_elect_epoch) >= 2:
                        offsets_slope_elect_all_epoch.append(np.polyfit(x, offsets_window_all_elect_epoch, 1)[0])
                    else:
                        offsets_slope_elect_all_epoch.append(np.nan)
                    offsets_std_elect_all_epoch.append(np.std(offsets_window_all_elect_epoch))

                    # exponents
                    x = range(len(exponents_window_all_elect_epoch))
                    if len(exponents_window_all_elect_epoch) >= 2:
                        exponents_slope_elect_all_epoch.append(np.polyfit(x, exponents_window_all_elect_epoch, 1)[0])
                    else:
                        exponents_slope_elect_all_epoch.append(np.nan)
                    exponents_std_elect_all_epoch.append(np.std(exponents_window_all_elect_epoch))


                    # ===== knee: solo si existe (valor + slope + std) =====
                    if aperiodic_mode == "knee":
                        
                        x = range(len(knee_window_all_elect_epoch))
                        if len(knee_window_all_elect_epoch) >= 2:
                            knee_slope_elect_all_epoch.append(
                                np.polyfit(x, knee_window_all_elect_epoch, 1)[0]
                            )
                        else:
                            knee_slope_elect_all_epoch.append(np.nan)
                        knee_std_elect_all_epoch.append(np.std(knee_window_all_elect_epoch))

                    
                ## ADD RESULT TO ALL EPOCHS  (FOOOF)
                ## ===== ADD RESULT TO ALL EPOCHS — VALORES POR VENTANA =====

                # ----- BANDAS -----
                delta_window_all_elect_all_epoch_all.append(delta_window_all_elect_all_epoch)
                theta_window_all_elect_all_epoch_all.append(theta_window_all_elect_all_epoch)
                alpha_window_all_elect_all_epoch_all.append(alpha_window_all_elect_all_epoch)
                beta_window_all_elect_all_epoch_all.append(beta_window_all_elect_all_epoch)
                gamma_window_all_elect_all_epoch_all.append(gamma_window_all_elect_all_epoch)

                # ----- APERIODIC -----
                offsets_window_all_elect_all_epoch_all.append(offsets_window_all_elect_all_epoch)
                exponents_window_all_elect_all_epoch_all.append(exponents_window_all_elect_all_epoch)

                # ----- MÉTRICAS DEL AJUSTE (valor por ventana) -----
                r2_window_all_elect_all_epoch_all.append(r2_window_all_elect_all_epoch)
                error_window_all_elect_all_epoch_all.append(error_window_all_elect_all_epoch)

                # ----- KNEE: valor por ventana -----
                if aperiodic_mode == "knee":
                    knee_window_all_elect_all_epoch_all.append(knee_window_all_elect_all_epoch)
                    
                    
                    
                ## ===== ADD RESULT TO ALL EPOCHS — SLOPE / STD =====

                ## ===== ADD RESULT TO ALL EPOCHS — SLOPE / STD =====

                # ----- BANDAS -----
                delta_slope_elect_all_epoch_all.append(delta_slope_elect_all_epoch)
                delta_std_elect_all_epoch_all.append(delta_std_elect_all_epoch)
                del delta_slope_elect_all_epoch, delta_std_elect_all_epoch

                theta_slope_elect_all_epoch_all.append(theta_slope_elect_all_epoch)
                theta_std_elect_all_epoch_all.append(theta_std_elect_all_epoch)
                del theta_slope_elect_all_epoch, theta_std_elect_all_epoch

                alpha_slope_elect_all_epoch_all.append(alpha_slope_elect_all_epoch)
                alpha_std_elect_all_epoch_all.append(alpha_std_elect_all_epoch)
                del alpha_slope_elect_all_epoch, alpha_std_elect_all_epoch

                beta_slope_elect_all_epoch_all.append(beta_slope_elect_all_epoch)
                beta_std_elect_all_epoch_all.append(beta_std_elect_all_epoch)
                del beta_slope_elect_all_epoch, beta_std_elect_all_epoch

                gamma_slope_elect_all_epoch_all.append(gamma_slope_elect_all_epoch)
                gamma_std_elect_all_epoch_all.append(gamma_std_elect_all_epoch)
                del gamma_slope_elect_all_epoch, gamma_std_elect_all_epoch


                # ----- APERIODIC -----
                offsets_slope_elect_all_epoch_all.append(offsets_slope_elect_all_epoch)
                offsets_std_elect_all_epoch_all.append(offsets_std_elect_all_epoch)
                del offsets_slope_elect_all_epoch, offsets_std_elect_all_epoch

                exponents_slope_elect_all_epoch_all.append(exponents_slope_elect_all_epoch)
                exponents_std_elect_all_epoch_all.append(exponents_std_elect_all_epoch)
                del exponents_slope_elect_all_epoch, exponents_std_elect_all_epoch


                # ----- KNEE: slope / std -----
                if aperiodic_mode == "knee":
                    knee_slope_elect_all_epoch_all.append(knee_slope_elect_all_epoch)
                    knee_std_elect_all_epoch_all.append(knee_std_elect_all_epoch)
                    del knee_slope_elect_all_epoch, knee_std_elect_all_epoch
                    
            ### end of loops


            # parameters of table
            num_elects=len(channels_mag)
            num_epochs=len(data_epochs)
            shape_tabla_dynamic=num_epochs*num_elects*window_number
            shape_tabla_results=num_epochs*num_elects

            

            

            # assert len(delta_window_all_elect_all_epoch_all) == num_epochs
            # assert len(delta_window_all_elect_all_epoch_all[0]) == num_elects
            # assert len(delta_window_all_elect_all_epoch_all[0][0]) == window_number
            
            # # creation of tables
            
            with open(log_file, "a") as f:
                f.write(f"{subj}: dynamic FOOOF completed\n")
        
            table_dynamic_fooof_cond = pd.DataFrame({
                'Subject': [subj] * shape_tabla_dynamic,
                'Condition': [cond_clean]*shape_tabla_dynamic,
                'Epoch': np.repeat(np.arange(num_epochs), num_elects * window_number),
                'Elect': np.tile(np.repeat(channels_mag, window_number), num_epochs),
                'Window': np.tile(np.arange(window_number), num_epochs * num_elects),

                # bandas
                'delta': np.array(delta_window_all_elect_all_epoch_all).ravel(),
                'theta': np.array(theta_window_all_elect_all_epoch_all).ravel(),
                'alpha': np.array(alpha_window_all_elect_all_epoch_all).ravel(),
                'beta':  np.array(beta_window_all_elect_all_epoch_all).ravel(),
                'gamma': np.array(gamma_window_all_elect_all_epoch_all).ravel(),

                # aperiodic
                'offsets': np.array(offsets_window_all_elect_all_epoch_all).ravel(),
                'exponents': np.array(exponents_window_all_elect_all_epoch_all).ravel(),

                # métricas (solo valor)
                'r2': np.array(r2_window_all_elect_all_epoch_all).ravel(),
                'error': np.array(error_window_all_elect_all_epoch_all).ravel(),
            })

            # knee solo si existe
            if aperiodic_mode == "knee":
                table_dynamic_fooof_cond['knee'] = np.array(
                    knee_window_all_elect_all_epoch_all
                ).ravel()



            #save table dynamic path
            table_dynamic_path = log_dir / f"{subj}_{cond}_table_dynamic_fooof_all_{suffix_str}.pickle"
            table_dynamic_fooof_cond.to_pickle(table_dynamic_path)
            del table_dynamic_fooof_cond
            
            table_dynamic_fooof_results_cond = pd.DataFrame({
                'Subject': [subj] * shape_tabla_results,
                'Condition': [cond_clean] * shape_tabla_results,
                'Epoch': np.repeat(np.arange(num_epochs), num_elects),
                'Elect': np.tile(channels_mag, num_epochs),

                # slopes de bandas
                'delta_slope': np.array(delta_slope_elect_all_epoch_all).ravel(),
                'theta_slope': np.array(theta_slope_elect_all_epoch_all).ravel(),
                'alpha_slope': np.array(alpha_slope_elect_all_epoch_all).ravel(),
                'beta_slope':  np.array(beta_slope_elect_all_epoch_all).ravel(),
                'gamma_slope': np.array(gamma_slope_elect_all_epoch_all).ravel(),

                # std de bandas
                'delta_std': np.array(delta_std_elect_all_epoch_all).ravel(),
                'theta_std': np.array(theta_std_elect_all_epoch_all).ravel(),
                'alpha_std': np.array(alpha_std_elect_all_epoch_all).ravel(),
                'beta_std':  np.array(beta_std_elect_all_epoch_all).ravel(),
                'gamma_std': np.array(gamma_std_elect_all_epoch_all).ravel(),

                # slopes aperiodic
                'offsets_slope': np.array(offsets_slope_elect_all_epoch_all).ravel(),
                'exponents_slope': np.array(exponents_slope_elect_all_epoch_all).ravel(),

                # std aperiodic
                'offsets_std': np.array(offsets_std_elect_all_epoch_all).ravel(),
                'exponents_std': np.array(exponents_std_elect_all_epoch_all).ravel(),
            })
            
            if aperiodic_mode == "knee":
                table_dynamic_fooof_results_cond['knee_slope'] = np.array(
                    knee_slope_elect_all_epoch_all
                ).ravel()
                table_dynamic_fooof_results_cond['knee_std'] = np.array(
                    knee_std_elect_all_epoch_all
                ).ravel()
            
            # save table dynamic foof results 
            table_dynamic_results_path = log_dir / f"{subj}_{cond}_table_dynamic_fooof_results_all_{suffix_str}.pickle"
            table_dynamic_fooof_results_cond.to_pickle(table_dynamic_results_path)
            del table_dynamic_fooof_results_cond
            
                
            del delta_slope_elect_all_epoch_all,delta_std_elect_all_epoch_all, theta_slope_elect_all_epoch_all, theta_std_elect_all_epoch_all, alpha_slope_elect_all_epoch_all, alpha_std_elect_all_epoch_all, beta_slope_elect_all_epoch_all,beta_std_elect_all_epoch_all, gamma_slope_elect_all_epoch_all,gamma_std_elect_all_epoch_all,offsets_slope_elect_all_epoch_all,offsets_std_elect_all_epoch_all,exponents_slope_elect_all_epoch_all,exponents_std_elect_all_epoch_all
            del delta_window_all_elect_all_epoch_all,theta_window_all_elect_all_epoch_all,alpha_window_all_elect_all_epoch_all,beta_window_all_elect_all_epoch_all,gamma_window_all_elect_all_epoch_all,offsets_window_all_elect_all_epoch_all,exponents_window_all_elect_all_epoch_all
            gc.collect()
            
            
            
            with open(log_file, "a") as f:
                f.write(f"{subj}_{cond}: tables saved\n")
                
        with open(log_file, "a") as f:
            f.write(f"{subj}: ALL_CONDITIONS_PROCESSED_OK\n")
               
        del epochs
            
    except Exception as e:
        with open(log_file, "a") as f:
            f.write(f"{subj}: ERROR -> {repr(e)}\n")

        error_log_file = log_dir / f"ERROR_{subj}_log_dynamic_fooof_{suffix_str}.txt"

        # Si ya existe un ERROR_..., lo sobreescribimos
        if error_log_file.exists():
            error_log_file.unlink()

        os.replace(log_file, error_log_file)  # rename/move atómicamente en Windows


      

    gc.collect()






In [8]:
#table_dynamic_autocorrelation

In [9]:
# canal_buscado = 'MLF31-4304'

# if canal_buscado in channels_mag:
#     indice = channels_mag.index(canal_buscado)
#     print(f"El canal {canal_buscado} está en la posición {indice}.")
# else:
#     print(f"El canal {canal_buscado} NO está en la lista de channels_mag.")

In [10]:
selected_subjects = [s for s in subjects if s not in subjects_remove]

selected_subjects

['sub-V1001',
 'sub-V1003',
 'sub-V1004',
 'sub-V1005',
 'sub-V1007',
 'sub-V1008',
 'sub-V1009',
 'sub-V1011',
 'sub-V1012',
 'sub-V1013',
 'sub-V1015',
 'sub-V1016',
 'sub-V1019',
 'sub-V1020',
 'sub-V1022',
 'sub-V1024',
 'sub-V1025',
 'sub-V1027',
 'sub-V1028',
 'sub-V1029',
 'sub-V1030',
 'sub-V1031',
 'sub-V1032',
 'sub-V1033',
 'sub-V1034',
 'sub-V1035',
 'sub-V1036',
 'sub-V1037',
 'sub-V1038',
 'sub-V1039',
 'sub-V1040',
 'sub-V1042',
 'sub-V1044',
 'sub-V1045',
 'sub-V1046',
 'sub-V1048',
 'sub-V1049',
 'sub-V1050',
 'sub-V1052',
 'sub-V1053',
 'sub-V1054',
 'sub-V1055',
 'sub-V1057',
 'sub-V1058',
 'sub-V1059',
 'sub-V1061',
 'sub-V1062',
 'sub-V1063',
 'sub-V1065',
 'sub-V1066',
 'sub-V1068',
 'sub-V1069',
 'sub-V1070',
 'sub-V1071',
 'sub-V1072',
 'sub-V1073',
 'sub-V1074',
 'sub-V1075',
 'sub-V1076',
 'sub-V1077',
 'sub-V1079',
 'sub-V1080',
 'sub-V1081',
 'sub-V1083',
 'sub-V1084',
 'sub-V1085',
 'sub-V1086',
 'sub-V1087',
 'sub-V1088',
 'sub-V1089',
 'sub-V1090',
 'sub-

In [11]:
# # # EXPECTED_CONDITIONS = {
# # #     # ---- ZINNEN ----
# # #     "begin_zinnen_RC_plus",
# # #     "begin_zinnen_RC_neg",
# # #     "begin_zinnen_RC_plus_question_hit",
# # #     "begin_zinnen_RC_plus_question_incorrect",
# # #     "begin_zinnen_RC_neg_question_hit",
# # #     "begin_zinnen_RC_neg_question_incorrect",

# # #     # ---- WOORDEN ----
# # #     "begin_woorden_RC_plus",
# # #     "begin_woorden_RC_neg",
# # #     "begin_woorden_RC_plus_question_hit",
# # #     "begin_woorden_RC_plus_question_incorrect",
# # #     "begin_woorden_RC_neg_question_hit",
# # #     "begin_woorden_RC_neg_question_incorrect",
# # # }

In [12]:
import os
from collections import defaultdict

FINISHED_RE = "ALL_CONDITIONS_PROCESSED_OK"

def compute_and_clean_subjects(
    selected_subjects,
    log_dir,
):
    files = os.listdir(log_dir)

    subj_log = {}

    # --- indexar logs (1 por sujeto)
    for f in files:
        if not f.endswith(".txt"):
            continue
        for subj in selected_subjects:
            if f.startswith(subj + "_"):
                subj_log[subj] = f
                break
            

        
    # --- 2️⃣ detectar sujetos con ERROR_*.txt
    subjects_error = set()

    for f in files:
        if f.startswith("ERROR_") and f.endswith(".txt"):
            # formato: ERROR_<subj>_log_...
            parts = f.split("_")
            if len(parts) > 1:
                subjects_error.add(parts[1])        
                

    subjects_processed_finished = []
    subjects_incomplete = []
    subjects_no_processed = []
    


    # --- clasificar sujetos
    for subj in selected_subjects:
        
        if subj in subjects_error:
            continue

        # nunca se empezó
        if subj not in subj_log:
            subjects_no_processed.append(subj)
            continue

        path = os.path.join(log_dir, subj_log[subj])
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            content = fh.read()

        
        if FINISHED_RE in content:
            subjects_processed_finished.append(subj)
            
        # not completed, but not wrong, it will be added to subjects_to process
        else:
            subjects_incomplete.append(subj)
            


    # --- limpiar SOLO sujetos incompletos
    for subj in subjects_error:
        print(f"[CLEANUP] Eliminando archivos de: {subj}")
        for f in files:
            if (
                (f.startswith(subj + "_") or f.startswith(f"ERROR_{subj}_"))
                and (f.endswith(".pickle") or f.endswith(".txt"))):      
                try:          
                    os.remove(os.path.join(log_dir, f))
                    print(f"    - eliminado: {f}")
                except FileNotFoundError:
                    pass

    subjects_to_process = sorted(
        set(subjects_incomplete) | set(subjects_no_processed) | set (subjects_error)
    )
    
    print("Subjects incomplete")
    for subj in subjects_incomplete:
        print(f"  - {subj}")
    
        
        
    print("Subjects to process:")
    for subj in subjects_to_process:
        print(f"  - {subj}")

    return (
        subjects_processed_finished,
        subjects_incomplete,
        subjects_no_processed,
        subjects_to_process,
        subjects_error
    )


In [13]:
(
    subjects_processed_finished,
    subjects_incomplete,
    subjects_no_processed,
    subjects_to_process,
    subjects_error
) = compute_and_clean_subjects(
    selected_subjects,
    log_dir
)

Subjects incomplete
Subjects to process:


In [14]:
len(selected_subjects)

95

In [15]:
len(subjects_to_process)

0

In [16]:



# dynamic_FOOOF_epochs(subj,window_size,sliding_window, bands,dict_select_channels,lfreq, hfreq,change_name=None ,aperiodic_mode="fixed", select_highest=False, select_channels=None,  filtering=False, filter_name=None)

Parallel(
    n_jobs=8,
    verbose=10,
)(
    delayed(dynamic_FOOOF_epochs)(subj,window_size,sliding_window, bands=freq_bands,dict_select_channels=select_channels,lfreq=lfreq, hfreq=hfreq,change_name=change_name ,aperiodic_mode=aperiodic_mode, select_highest=select_highest, select_channels=select_channels,  filtering=filtering, filter_name=filter_name)
    for subj in subjects_to_process
)

print("Dynamic fooof completed")


Dynamic fooof completed


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


In [17]:
# # -------------------------------
# # Leer tablas por sujeto
# # -------------------------------
# all_tables_dynamic = []
# all_tables_dynamic_results = []

# for subj in selected_subjects:
#     path_dyn = log_dir / f"{subj}_table_dynamic_fooof_all_{suffix_str}.pickle"
#     path_res = log_dir / f"{subj}_table_dynamic_fooof_results_all_{suffix_str}.pickle"14

#     all_tables_dynamic.append(pd.read_pickle(path_dyn))
#     all_tables_dynamic_results.append(pd.read_pickle(path_res))

# # -------------------------------
# # Concatenar
# # -------------------------------
# table_dynamic_fooof_all = pd.concat(all_tables_dynamic, ignore_index=True)
# table_dynamic_fooof_results_all = pd.concat(all_tables_dynamic_results, ignore_index=True)

# # -------------------------------
# # Guardar finales
# # -------------------------------
# table_dynamic_path = ACW_path / f"table_dynamic_fooof_all_{suffix_str}.pickle"
# table_dynamic_results_path = ACW_path / f"table_dynamic_fooof_results_all_{suffix_str}.pickle"

# table_dynamic_fooof_all.to_pickle(table_dynamic_path)
# print(f"Saved {table_dynamic_path.name}")

# table_dynamic_fooof_results_all.to_pickle(table_dynamic_results_path)
# print(f"Saved {table_dynamic_results_path.name}")

# 50bn0 m,

In [ ]:
for subj in selected_subjects:
    for cond in 

In [18]:
def validar_dynamic_tables(dynamic_df, results_df):
    print("\n==============================")
    print("🔎 VALIDACIÓN TABLA DINÁMICA (con ventanas)")
    print("==============================")

    # 🔧 Obtener número esperado de canales automáticamente desde la tabla de resultados
    channels_por_epoch = results_df.groupby(["Subject", "Condition", "Epoch"])["Elect"].count()
    expected_channels = channels_por_epoch.mode().iloc[0]  # Valor más frecuente
    print(f"✅ Número esperado de canales detectado automáticamente: {expected_channels}\n")

    # Validar número de canales por epoch-window-condición
    counts_dyn = dynamic_df.groupby(["Subject", "Condition", "Epoch", "Window"])["Elect"].count().reset_index()
    counts_dyn.rename(columns={"Elect": "NumCanales"}, inplace=True)

    print("📌 Distribución de canales por ventana (detectado en dynamic_df):")
    print(counts_dyn["NumCanales"].value_counts(), "\n")

    if len(counts_dyn["NumCanales"].value_counts()) == 1 and counts_dyn["NumCanales"].iloc[0] == expected_channels:
        print(f"✅ Todas las ventanas tienen exactamente {expected_channels} canales.")
    else:
        print("⚠ Atención: se detectaron ventanas con conteos distintos de canales.")
        inconsist_dyn = counts_dyn[counts_dyn["NumCanales"] != expected_channels]
        print("🔬 Ventanas inconsistentes:")
        print(inconsist_dyn.to_string(index=False))

    print("\n==============================")
    print("🔎 VALIDACIÓN TABLA DE RESULTADOS (sin ventanas)")
    print("==============================")

    counts_res = results_df.groupby(["Subject", "Condition", "Epoch"])["Elect"].count().reset_index()
    counts_res.rename(columns={"Elect": "NumCanales"}, inplace=True)

    print("📌 Distribución de canales por época (detectado en results_df):")
    print(counts_res["NumCanales"].value_counts(), "\n")

    if len(counts_res["NumCanales"].value_counts()) == 1 and counts_res["NumCanales"].iloc[0] == expected_channels:
        print(f"✅ Todas las épocas tienen exactamente {expected_channels} canales.")
    else:
        print("⚠ Atención: se detectaron épocas con conteos distintos de canales.")
        inconsist_res = counts_res[counts_res["NumCanales"] != expected_channels]
        print("🔬 Épocas inconsistentes:")
        print(inconsist_res.to_string(index=False))

    print("\n==============================")
    print("🔍 VALIDACIÓN DE CONSISTENCIA DE CANALES ENTRE SUJETOS")
    print("==============================")

    channels_por_sujeto = results_df.groupby("Subject")["Elect"].unique()
    base_channels = set(channels_por_sujeto.iloc[0])
    consistentes = True
    for sujeto, canales in channels_por_sujeto.items():
        if set(canales) != base_channels:
            consistentes = False
            print(f"⚠ Diferencias en {sujeto}: {set(canales) ^ base_channels}")

    if consistentes:
        print("✅ Todos los sujetos tienen exactamente los mismos canales.")
    else:
        print("⚠ Hay sujetos con diferentes listas de canales. Revisar detalles arriba.")

    print("\n==============================")
    print("🔍 Validación completada")
    print("==============================")

In [19]:
# validar_dynamic_tables(table_dynamic_fooof_all, table_dynamic_fooof_results_all)


In [ ]:


# =========================================================
# 1) UNIR table_dynamic_fooof_results
# =========================================================

pattern_results = f"*_table_dynamic_fooof_results_all_{suffix_str}.pickle"
files_results = sorted(log_dir.glob(pattern_results))

dfs_results = []

for fp in files_results:
    try:
        df = pd.read_pickle(fp)
        dfs_results.append(df)
        del df
    except Exception as e:
        print(f"[ERROR] No se pudo leer {fp.name}: {e}")

if len(dfs_results) == 0:
    raise RuntimeError("No se pudo cargar ningún pickle de RESULTS")

table_dynamic_fooof_results_all = pd.concat(
    dfs_results, ignore_index=True, sort=False
)

out_results = ACW_path / f"table_dynamic_fooof_results_all_{suffix_str}.pickle"
table_dynamic_fooof_results_all.to_pickle(out_results)

print(
    f"[OK] Guardado {out_results.name} | "
    f"shape={table_dynamic_fooof_results_all.shape} | "
    f"n_files={len(dfs_results)}"
)

# =========================================================
# 2) UNIR table_dynamic_fooof
# =========================================================

pattern_main = f"*_table_dynamic_fooof_all_{suffix_str}.pickle"
files_main = sorted(log_dir.glob(pattern_main))

dfs_main = []

for fp in files_main:
    try:
        df = pd.read_pickle(fp)
        dfs_main.append(df)
        del df
    except Exception as e:
        print(f"[ERROR] No se pudo leer {fp.name}: {e}")

if len(dfs_main) == 0:
    raise RuntimeError("No se pudo cargar ningún pickle de MAIN")

table_dynamic_fooof_all = pd.concat(
    dfs_main, ignore_index=True, sort=False
)

out_main = ACW_path / f"table_dynamic_fooof_all_{suffix_str}.pickle"
table_dynamic_fooof_all.to_pickle(out_main)

print(
    f"[OK] Guardado {out_main.name} | "
    f"shape={table_dynamic_fooof_all.shape} | "
    f"n_files={len(dfs_main)}"
)


[OK] Guardado table_dynamic_fooof_results_all_filt_1-40_event.pickle | shape=(6442470, 18) | n_files=1096
[OK] Guardado table_dynamic_fooof_all_filt_1-40_event.pickle | shape=(96637050, 14) | n_files=1096
